In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.io
import seaborn as sns
import networkx as nx
import bct
import scipy.stats as stats
from pymatreader import read_mat
from proc_funcs import calculate_graph_metrics

Notes on harmonized data: all 4216 subjects were included in harmonization. then density outliers were removed (4202). Supplementary Fig 1 to see. 

Variable density: what we will use finally, it takes out by absolute threshod 30 of the data taking as noise the lower stream counts. Supplementary fig 1a: 

Generalized additive model of unthresholded network density (‘Raw’) while controlling for sex, dataset, atlas, and neurodiversity group (neurotypical or neurodiverse). This regression was reduced by 30% (multiplied by 0.70), which was used to get age-specific target densities for thresholding (‘Reduced’). Violin plots show examples of density for age bins 0, 30, 60, and 90 years before thresholding (black) and after thresholding to target density (blue).

In the methods, thersholding section: For the variable density analysis, we performed a generalized additive model (see “Methods”, “Statistics”) on the raw network densities and took 70% of the regression to obtain a ‘target’ density for each age (Supplementary Fig. 1a). Then, for each age group within each study, we applied the absolute threshold based on streamline count cut-off that yielded an average density equal to the target density for that age. The resulting networks were thresholded to densities ranging from 21 to 8%, with the original relationship between age and density preserved. 

In supplementary Fig 3 there is all significance with sex for controlled density networks. In replication_mousley_age_sex_pval.R we reproduce that exact p val with our her and ours revealing the exact processing pipeline. 
 
In the variable density pipeline there are other turning points identified: 9,39,84. authors anyway cap turning point analysis at 90, so in both cases the last epoch goes till 90 (supplemntray fig 10.d). In any case, authors always use turning points identified for controlled density. 

In [2]:
demo = pd.read_csv('preprocdata/demographics_harm.csv').reset_index(drop=True).drop(columns='remove_index') #density outliers and non neurotypical have been removed.

#puedo usar neuroCombat o coger directamente los datos de la autora harmonizados.
#Covariates are age, sex. harmonization is done for atlas and dataset.
from pymatreader import read_mat
connectomes_vd = read_mat('preprocdata/harmonized_data.mat')['harmonized_data']['connectomes']['variable_density']['normalized_weighted']
connectomes_cd = read_mat('preprocdata/harmonized_data.mat')['harmonized_data']['connectomes']['controlled_density']['normalized_weighted']


c:\Users\Ale\miniconda3\envs\snakes\Lib\site-packages\pymatreader\utils.py:304: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(


In [3]:
metrics_df = calculate_graph_metrics(connectomes_cd, connectomes_vd) #variable density connectomes used only for strength and disparity.  
#Controlled density strength is also used for the GAM age-strength GAM comparisson. 
data=pd.DataFrame({'Subject': np.arange(metrics_df.shape[0])})
data=pd.concat([data,demo], axis=1)
data=pd.concat([data, metrics_df], axis=1).to_csv('procdata/demographics_with_graph_metrics.csv', index=False)

In [3]:
data=pd.read_csv('procdata/demographics_with_graph_metrics.csv')
print(data.head())

   Subject  age  sex  dataset  atlas  Strength  Strength_CD  Disparity_Y  \
0        0    0    0        1      1  0.477385     0.461686     0.235901   
1        1    0    0        1      1  0.444135     0.423765     0.221756   
2        2    0    0        1      1  0.697265     0.653556     0.203551   
3        3    0    1        1      1  0.981152     0.946121     0.211505   
4        4    0    0        1      1  0.613663     0.583832     0.233065   

   Clustering_Coefficient  Average_Shortest_Path  Global_Efficiency  \
0                0.015168               2.485699           0.464016   
1                0.013921               2.414595           0.472454   
2                0.018867               2.418757           0.470563   
3                0.028460               2.561615           0.455707   
4                0.017842               2.492850           0.460921   

       Diameter  Modularity   Density  
0  1.000000e+20    0.499730  0.100125  
1  1.000000e+20    0.449409  0.10037

Now we try to reproduce strength-age correlation for different epochs (Tab. 2 for r values and and Suppl table 4 for p-vals) using GAM results of replication_mousley_age_epochs.R to get the residuals and Python to analyze in this script. 

In [10]:
input_data = pd.read_csv('procdata/strength_residuals.csv') #GAM residuals for strength calculated in R. Calculated in replication_muosley_age_epochs.R

# Epochs: inclusive bounds. 
epoch_ranges = [
    (0, 9),
    (9, 32),
    (32, 66),
    (66, 83),
    (83, 90)
]

rows = []

# Replicación exacta del bucle de la autora (usando input_data, NO grouped_data)
for epoch_index, (start, end) in enumerate(epoch_ranges):
    # Filtrar sujeto a sujeto
    epoch_data = input_data[(input_data['age'] >= start) & (input_data['age'] <= end)]
    
    # Extraer variables (col 1 es strength, col 0 es age)
    y = epoch_data.iloc[:, 0]  # Age
    measure = epoch_data.iloc[:, 1]  # Strength (residuos)
    
    if len(epoch_data) > 2:
        r, p = stats.pearsonr(measure, y)
        rows.append({
            "Epoch": epoch_index + 1,
            "Age Range": f"{start}-{end}",
            "N_subjects": epoch_data.shape[0],
            "r": round(r, 4),
            "p": p
        })

df_results = pd.DataFrame(rows)
#display(df_results)


In [11]:
input_data.columns

Index(['age', 'strength'], dtype='object')

In [5]:
data.columns

Index(['Subject', 'age', 'sex', 'dataset', 'atlas', 'Strength', 'Strength_CD',
       'Disparity_Y', 'Clustering_Coefficient', 'Average_Shortest_Path',
       'Global_Efficiency', 'Diameter', 'Modularity'],
      dtype='object')

In [7]:
# Now we generate distribution plots for each metric, stored in a plots folder.
metrics_to_plot = ['Strength', 'Clustering_Coefficient', 'Global_Efficiency', 'Modularity','Diameter','Disparity_Y']
for metric in metrics_to_plot:
    plt.figure(figsize=(10, 6))
    sns.histplot(data[metric], kde=True, bins=30)
    plt.title(f'Distribution of {metric.capitalize()}')
    plt.xlabel(metric.capitalize())
    plt.ylabel('Frequency')
    plt.savefig(f'results/data_distributions/{metric}_distribution.png')
    plt.close()